# Cobertura y Conectividad Móvil en México (geodatos con granularidad municipal)

### Fuentes
1. **Ookla:**
   - **Descripción:** Datos de velocidad y conectividad móvil.

2. **Marco Geoestadístico del INEGI:**
   - **Descripción:** Límites municipales y estatales de México.

### Diccionario: Variables creadas
- **`avg_d_mbps`:** Velocidad promedio de descarga en Mbps.
- **`tech_proxy`:** Tecnología predominante en el área (3G, 4G, 5G).
- **`int_pct_3g_coverage`:** Porcentaje promedio de cobertura 3G.
- **`int_pct_4g_coverage`:** Porcentaje promedio de cobertura 4G.
- **`int_pct_5g_coverage`:** Porcentaje promedio de cobertura 5G.

### Raw: Archivos originales
  - `00mun.shp`: Límites municipales.
  - `00est.shp`: Límites estatales.
  - `gps_mobile_tiles.shp`: Hexágonos de conectividad móvil.

### Proxy: Metodología para determinar la categoría
- **Definición:** La categoría de tecnología (`tech_proxy`) se determina en función de la velocidad promedio de descarga:
  - **3G:** Velocidad menor a 10 Mbps.
  - **4G:** Velocidad entre 10 y 40 Mbps.
  - **5G:** Velocidad mayor a 40 Mbps.
- **Justificación:** Este proxy se basa en estándares generales (usados por el IFT para sus propios mapas) de capacidad de red para cada tecnología.

### Filtro de confianza
- **Definición:** Se filtran los hexágonos con menos de `n` pruebas realizadas.
- **Justificación:** Este filtro asegura que los datos utilizados sean representativos y confiables, eliminando áreas con poca información que podrían sesgar los resultados.


In [58]:
import geopandas as gpd
import pandas as pd
import os

## Carga de shapefiles

In [ ]:
path = os.path.join('..', 'data', 'raw')

# Verificar si los archivos existen
municipios_path = os.path.join(path, '00mun.shp')
ookla_path = os.path.join(path, 'gps_mobile_tiles.shp')

municipios = gpd.read_file(municipios_path)
ookla_global = gpd.read_file(ookla_path)

In [77]:
# Corroborar CRS
if municipios.crs != "EPSG:4326":
    municipios = municipios.to_crs("EPSG:4326")
ookla_global = ookla_global.to_crs("EPSG:4326")

### CRS: Proyecciones usadas
- **Proyección inicial:** Los datos de los shapefiles (`00mun.shp`, `00est.shp` y `gps_mobile_tiles.shp`) se cargan con sus sistemas de referencia espacial originales.
- **Proyección utilizada:** Se reproyectan al sistema de coordenadas EPSG:3857 (Web Mercator) para realizar operaciones espaciales como el `Spatial Join`. Finalmente, los datos se transforman a EPSG:4326 (WGS 84) para exportar los resultados en un formato estándar compatible con herramientas de visualización y análisis.

## Filtrado México

In [78]:
# Bounding Box México
minx, miny, maxx, maxy = -118.5, 14.5, -86.5, 32.7
ookla_mexico = ookla_global.cx[minx:maxx, miny:maxy].copy()

ookla_mexico.shape

(103505, 7)

In [79]:
# Spatial Join (usando centroide)
ookla_mexico['geometry_backup'] = ookla_mexico.geometry
ookla_mexico['geometry'] = ookla_mexico.geometry.centroid

# Cruce con municipios
ookla_final = gpd.sjoin(ookla_mexico, municipios, how="inner", predicate="within")

# Restauración de la geometría de hexágono para el dataset granular
ookla_final['geometry'] = ookla_final['geometry_backup']
ookla_final = ookla_final.to_crs("EPSG:4326")

## Limpieza e intervalos (inferencia de tecnología)

In [ ]:
ookla_final['avg_d_mbps'] = ookla_final['avg_d_kbps'] / 1000

# Clasificación por tecnología (Proxy de capacidad de red)
def clasificar_tech(speed):
    if speed < 10: return '3G'
    if speed < 40: return '4G'
    return '5G'

ookla_final['tech_proxy'] = ookla_final['avg_d_mbps'].apply(clasificar_tech)

## Generación de datasets (municipal y estatal)

In [82]:
# Calcular porcentajes de cobertura para cada tecnología
ookla_final['int_pct_3g_coverage'] = ookla_final['tech_proxy'].apply(lambda x: 100 if x == '3G' else 0)
ookla_final['int_pct_4g_coverage'] = ookla_final['tech_proxy'].apply(lambda x: 100 if x == '4G' else 0)
ookla_final['int_pct_5g_coverage'] = ookla_final['tech_proxy'].apply(lambda x: 100 if x == '5G' else 0)

# Agrupar por municipio
cobertura_por_municipio = ookla_final.groupby(['CVE_ENT', 'CVEGEO', 'NOMGEO']).agg({
    'avg_d_mbps': 'mean',  # Velocidad promedio
    'tech_proxy': lambda x: x.mode()[0],  # Tecnología predominante
    'geometry': 'count',  # Placeholder para calcular porcentajes
    'tests': 'sum',  # Total de tests
    'int_pct_3g_coverage': 'mean',  # Porcentaje promedio de cobertura 3G
    'int_pct_4g_coverage': 'mean',  # Porcentaje promedio de cobertura 4G
    'int_pct_5g_coverage': 'mean'   # Porcentaje promedio de cobertura 5G
}).reset_index()

# Renombrar columnas para cumplir con los requisitos
cobertura_por_municipio.rename(columns={
    'CVEGEO': 'id_cvegeo',
    'NOMGEO': 'nom_mun',
    'avg_d_mbps': 'int_avg_speed',
    'tech_proxy': 'int_tech_predominante',
    'int_pct_3g_coverage': 'int_pct_3g_coverage',
    'int_pct_4g_coverage': 'int_pct_4g_coverage',
    'int_pct_5g_coverage': 'int_pct_5g_coverage'
}, inplace=True)

# Dataset Estatal
cobertura_por_estado = cobertura_por_municipio.groupby(['CVE_ENT']).agg({
    'int_avg_speed': 'mean',
    'int_pct_3g_coverage': 'mean',
    'int_pct_4g_coverage': 'mean',
    'int_pct_5g_coverage': 'mean',
    'int_tech_predominante': lambda x: x.mode()[0]  # Tecnología predominante
}).reset_index()

## Exportación

In [83]:
cobertura_por_municipio.to_json('../data/processed/cobertura_red_por_municipio_2025.json')
cobertura_por_estado.to_json('../data/processed/cobertura_red_por_estado_2025.json')
# El dataset granular para mapas (solo México)
ookla_final[['tech_proxy', 'avg_d_mbps', 'geometry']].to_file('../data/processed/cobertura_mexico_geodatos_2025.geojson', driver='GeoJSON')

print("Datasets exportados.")

Datasets exportados.
